## **Celebal Excellence Internship Program 2026**
*Week 5 : Spark Assignment*

*Name: Manjit Bajaj*



## **Apache Spark Data Cleaning and DataFrame Analysis**

### **Objective**

This notebook demonstrates how Apache Spark DataFrames can be used to clean, transform, filter, and analyze dirty café transaction data.

### **1. Spark and MapReduce Overview**

MapReduce is a distributed data processing model that commonly performs intermediate processing using disk-based operations.

Apache Spark is faster for many workloads because it can keep intermediate data in memory. Spark also provides a high-level DataFrame API, which makes data cleaning and analysis easier.

Spark DataFrames are immutable. This means that transformations do not directly modify an existing DataFrame. Instead, they create a new DataFrame.

In [1]:
# import spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col,when,trim,max,min,count,sum,avg)
from pyspark.sql.types import IntegerType, DoubleType

In [2]:
# create sparksession
spark = SparkSession.builder.appName("Data Cleaning and Analysis").getOrCreate()
print("SparkSession created successfully.")

SparkSession created successfully.


### **2. Load the Dataset**

The dirty café sales dataset is loaded from the `data` folder into a Spark DataFrame.

The dataset contains inconsistent values such as `ERROR`, `UNKNOWN`, and missing values, making it suitable for data cleaning.

In [3]:
# load dataset
df = spark.read.csv("../data/dirty_cafe_sales.csv", header=True, inferSchema=True)
print("Dataset loaded Successfully!")

Dataset loaded Successfully!


In [4]:
df.show(10)

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_1961373|  Coffee|       2|           2.0|        4.0|   Credit Card|Takeaway|      2023-09-08|
|   TXN_4977031|    Cake|       4|           3.0|       12.0|          Cash|In-store|      2023-05-16|
|   TXN_4271903|  Cookie|       4|           1.0|      ERROR|   Credit Card|In-store|      2023-07-19|
|   TXN_7034554|   Salad|       2|           5.0|       10.0|       UNKNOWN| UNKNOWN|      2023-04-27|
|   TXN_3160411|  Coffee|       2|           2.0|        4.0|Digital Wallet|In-store|      2023-06-11|
|   TXN_2602893|Smoothie|       5|           4.0|       20.0|   Credit Card|    NULL|      2023-03-31|
|   TXN_4433211| UNKNOWN|       3|           3.0|        9.0|         ERR

### **3. Inspect the dataset**


In [5]:
# view columns names
print(df.columns)

['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']


In [6]:
# view data types
df.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Price Per Unit: string (nullable = true)
 |-- Total Spent: string (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: string (nullable = true)



In [7]:
# count total number of rows
print("Total number of rows:", df.count())

Total number of rows: 10000


In [8]:
# display summary statistics for numeric columns
df.describe().show()

+-------+--------------+-------+-----------------+------------------+-----------------+--------------+--------+----------------+
|summary|Transaction ID|   Item|         Quantity|    Price Per Unit|      Total Spent|Payment Method|Location|Transaction Date|
+-------+--------------+-------+-----------------+------------------+-----------------+--------------+--------+----------------+
|  count|         10000|   9667|             9862|              9821|             9827|          7421|    6735|            9841|
|   mean|          NULL|   NULL|3.028463396702027| 2.949984155487483|8.924352495262161|          NULL|    NULL|            NULL|
| stddev|          NULL|   NULL|1.419006873221319|1.2784504728035881|6.009919472829945|          NULL|    NULL|            NULL|
|    min|   TXN_1000555|   Cake|                1|               1.0|              1.0|          Cash|   ERROR|      2023-01-01|
|    max|   TXN_9999124|UNKNOWN|          UNKNOWN|           UNKNOWN|          UNKNOWN|       UNK

**Insight**

The dataset contains 10,000 transaction records. The first few rows, column names, schema, and summary statistics were inspected to understand the structure and quality of the dataset.

### **4. Check Dirty Data**

In [9]:
# Check null values
# This tell us how many NULL values exist in each column.
from pyspark.sql.functions import col, sum
null_counts_before = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts_before.show()


+--------------+----+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+----+--------+--------------+-----------+--------------+--------+----------------+
|             0| 333|     138|           179|        173|          2579|    3265|             159|
+--------------+----+--------+--------------+-----------+--------------+--------+----------------+



In [10]:
rows_before_duplicates = df.count()
df_no_duplicates = df.dropDuplicates()
rows_after_duplicates = df_no_duplicates.count()

print("Rows Before Removing Duplicates:", rows_before_duplicates)
print("Rows After Removing Duplicates:", rows_after_duplicates)
print("Duplicate Rows Removed:",rows_before_duplicates - rows_after_duplicates
)

Rows Before Removing Duplicates: 10000
Rows After Removing Duplicates: 10000
Duplicate Rows Removed: 0


**Insight**

Duplicate records were removed using `dropDuplicates()`. The row counts before and after the operation were compared to confirm whether duplicate records existed and were removed.

### **5. Handle Missing and Inconsistent Values**

The dataset contains:

- Actual NULL values
- `ERROR`
- `UNKNOWN`
- Empty or whitespace values

Instead of deleting a large number of rows, invalid values are first converted to NULL and then replaced with suitable values.

This approach preserves more transaction records.

In [11]:
df_clean = df_no_duplicates

In [12]:
text_columns = [
    "Item",
    "Payment Method",
    "Location"
]

numeric_columns = [
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

In [14]:
for column in text_columns:
    df_clean = df_clean.withColumn(
        column,
        when(
            trim(col(column)).isin("", "ERROR", "UNKNOWN", "NULL"),
            None
        ).otherwise(trim(col(column)))
    )

for column in numeric_columns:
    df_clean = df_clean.withColumn(
        column,
        when(
            trim(col(column)).isin("", "ERROR", "UNKNOWN", "NULL"),
            None
        ).otherwise(trim(col(column)))
    )

In [15]:
df_clean.show(10)

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_5307411|    Cake|       3|           3.0|       NULL|Digital Wallet|Takeaway|      2023-10-21|
|   TXN_3381656|    Cake|       5|           3.0|       15.0|          Cash|Takeaway|      2023-10-26|
|   TXN_6111710|   Salad|       3|           5.0|       15.0|   Credit Card|In-store|      2023-06-18|
|   TXN_8715122|    Cake|       2|           3.0|        6.0|          Cash|Takeaway|           ERROR|
|   TXN_1270603|   Juice|       4|           3.0|       12.0|   Credit Card|Takeaway|      2023-12-03|
|   TXN_9798525|  Coffee|       2|           2.0|        4.0|Digital Wallet|Takeaway|      2023-11-05|
|   TXN_9947479|Smoothie|       5|           4.0|       20.0|          Ca

**Insight**

Invalid values such as `ERROR`, `UNKNOWN`, and empty strings were converted to NULL. This standardizes inconsistent values and makes the dataset ready for type conversion and further cleaning.

In [16]:
df_clean = df_clean.fillna({
    "Item": "Unknown",
    "Payment Method": "Unknown",
    "Location": "Unknown",
    "Quantity": 0,
    "Price Per Unit": 0.0,
    "Total Spent": 0.0
})

In [17]:
print("Rows Before Cleaning:", df_no_duplicates.count())
print("Rows After Handling Missing Values:", df_clean.count())

Rows Before Cleaning: 10000
Rows After Handling Missing Values: 10000


In [15]:
# OR drop rows with null values
# df_no_null = df.dropna()

**Insight**

Initially, dropna() was tested, but it removed 5,450 out of 10,000 rows. Since this resulted in significant data loss, the fillna() method was selected. 

Missing values were handled using `fillna()`. The row count remained unchanged, showing that records were preserved during the cleaning process.
- Missing categorical values are replaced with `Unknown`.
- For numeric columns, missing values are temporarily replaced with `0` so that the dataset can be converted into a consistent numeric schema.
- The row count is preserved instead of deleting a large portion of the dataset.

### **7. Modify the Data Schema**

The dataset originally contains numeric columns that may be stored as strings because the original data contained invalid text values.

The following columns are converted to appropriate data types:

- Quantity → Integer
- Price Per Unit → Double
- Total Spent → Double

In [18]:
df_clean = df_clean.withColumn(
    "Quantity",
    col("Quantity").cast(IntegerType())
)

df_clean = df_clean.withColumn(
    "Price Per Unit",
    col("Price Per Unit").cast(DoubleType())
)

df_clean = df_clean.withColumn(
    "Total Spent",
    col("Total Spent").cast(DoubleType())
)

In [19]:
df_clean.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Item: string (nullable = false)
 |-- Quantity: integer (nullable = true)
 |-- Price Per Unit: double (nullable = true)
 |-- Total Spent: double (nullable = true)
 |-- Payment Method: string (nullable = false)
 |-- Location: string (nullable = false)
 |-- Transaction Date: string (nullable = true)



**Insight**

The schema was successfully modified. Numeric columns are now represented using suitable numeric data types, allowing mathematical operations such as average, minimum, maximum, and sum.

### **8. Rename Columns**

Column names are renamed to lowercase names with underscores.

This creates a cleaner and more consistent schema for data analysis.

In [20]:
for column in df_clean.columns:
    df_clean = df_clean.withColumnRenamed(
        column,
        column.lower().replace(" ", "_")
    )

print(df_clean.columns)

['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent', 'payment_method', 'location', 'transaction_date']


**Insight**

Column names were standardized using lowercase characters and underscores. This improves readability and makes column references easier when writing Spark code.

### **9. Filter Data**

Filtering is used to select records that satisfy a specific condition.


In [21]:
# filter by item
coffee_sales = df_clean.filter(
    col("item") == "Coffee"
)
coffee_sales.show(5)
print("Coffee Transactions:",coffee_sales.count())

+--------------+------+--------+--------------+-----------+--------------+--------+----------------+
|transaction_id|  item|quantity|price_per_unit|total_spent|payment_method|location|transaction_date|
+--------------+------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_9798525|Coffee|       2|           2.0|        4.0|Digital Wallet|Takeaway|      2023-11-05|
|   TXN_5125382|Coffee|       1|           2.0|        2.0|Digital Wallet|In-store|      2023-12-26|
|   TXN_2417721|Coffee|       4|           2.0|        8.0|   Credit Card|Takeaway|      2023-10-20|
|   TXN_4573096|Coffee|       1|           2.0|        2.0|Digital Wallet|In-store|      2023-08-13|
|   TXN_1256513|Coffee|       4|           2.0|        8.0|   Credit Card|In-store|      2023-02-21|
+--------------+------+--------+--------------+-----------+--------------+--------+----------------+
only showing top 5 rows
Coffee Transactions: 1165


In [24]:
# filter by payment method
credit_card_sales = df_clean.filter(
    col("payment_method") == "Credit Card"
)
print("Credit Card Transactions:", credit_card_sales.count())

Credit Card Transactions: 2273


In [25]:
# filter high-value transaction
high_value_sales = df_clean.filter(
    col("total_spent") > 10
)
high_value_sales.show(5)
print("Transactions Above 10:", high_value_sales.count())

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|transaction_id|    item|quantity|price_per_unit|total_spent|payment_method|location|transaction_date|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_3381656|    Cake|       5|           3.0|       15.0|          Cash|Takeaway|      2023-10-26|
|   TXN_6111710|   Salad|       3|           5.0|       15.0|   Credit Card|In-store|      2023-06-18|
|   TXN_1270603|   Juice|       4|           3.0|       12.0|   Credit Card|Takeaway|      2023-12-03|
|   TXN_9947479|Smoothie|       5|           4.0|       20.0|          Cash|Takeaway|      2023-09-27|
|   TXN_9555958|Smoothie|       4|           4.0|       16.0|Digital Wallet|Takeaway|      2023-09-30|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
only showing top 5 rows
Transactions Above 10: 3122


**Insight**

Filtering was used to identify specific subsets of the data. For example, transactions for Coffee, Credit Card payments, and transactions with spending greater than 10 were successfully identified.

### **10. Aggregation**

Spark aggregation functions are used to calculate summary statistics.

In [26]:
summary = df_clean.select(
    count("*").alias("total_rows"),
    sum("total_spent").alias("total_sales"),
    avg("total_spent").alias("average_spent"),
    min("total_spent").alias("minimum_spent"),
    max("total_spent").alias("maximum_spent")
)

summary.show()

+----------+-----------+-------------+-------------+-------------+
|total_rows|total_sales|average_spent|minimum_spent|maximum_spent|
+----------+-----------+-------------+-------------+-------------+
|     10000|    84763.5|      8.47635|          0.0|         25.0|
+----------+-----------+-------------+-------------+-------------+



**Insight**

Aggregation functions were used to calculate overall statistics for the dataset. These results provide a high-level view of transaction volume and spending behavior.

### **11. Group Data Using groupBy()**

The `groupBy()` operation groups records based on a column.


In [ ]:
# group by item
sales_by_item = df_clean.groupBy("item").agg(
    count("*").alias("transaction_count"),
    sum("total_spent").alias("total_sales"),
    avg("total_spent").alias("average_sales")
)
sales_by_item.show()

+--------+-----------------+-----------+------------------+
|    item|transaction_count|total_sales|     average_sales|
+--------+-----------------+-----------+------------------+
|   Salad|             1148|    16605.0|14.464285714285714|
|     Tea|             1089|     4735.5| 4.348484848484849|
|Sandwich|             1131|    12956.0|11.455349248452697|
| Unknown|              969|     8140.0| 8.400412796697626|
|   Juice|             1171|     9984.0|  8.52604611443211|
|Smoothie|             1096|    12556.0|11.456204379562044|
|  Coffee|             1165|     6784.0| 5.823175965665236|
|    Cake|             1139|     9933.0| 8.720807726075504|
|  Cookie|             1092|     3070.0| 2.811355311355311|
+--------+-----------------+-----------+------------------+



In [28]:
# group by payment method
sales_by_payment = df_clean.groupBy("payment_method").agg(
    count("*").alias("transaction_count"),
    sum("total_spent").alias("total_sales"),
    avg("total_spent").alias("average_sales")
)
sales_by_payment.show()

+--------------+-----------------+-----------+-----------------+
|payment_method|transaction_count|total_sales|    average_sales|
+--------------+-----------------+-----------+-----------------+
|   Credit Card|             2273|    19599.0|8.622525296964364|
|       Unknown|             3178|    26260.5|8.263215859030836|
|Digital Wallet|             2291|    19578.0| 8.54561326931471|
|          Cash|             2258|    19326.0|8.558901682905226|
+--------------+-----------------+-----------+-----------------+



In [29]:
# group by location
sales_by_location = df_clean.groupBy("location").agg(
    count("*").alias("transaction_count"),
    sum("total_spent").alias("total_sales")
)

sales_by_location.show()

+--------+-----------------+-----------+
|location|transaction_count|total_sales|
+--------+-----------------+-----------+
|In-store|             3017|    25906.0|
|Takeaway|             3022|    25229.5|
| Unknown|             3961|    33628.0|
+--------+-----------------+-----------+



In [30]:
sales_by_item.filter(col("total_sales") > 10000).show()

+--------+-----------------+-----------+------------------+
|    item|transaction_count|total_sales|     average_sales|
+--------+-----------------+-----------+------------------+
|   Salad|             1148|    16605.0|14.464285714285714|
|Sandwich|             1131|    12956.0|11.455349248452697|
|Smoothie|             1096|    12556.0|11.456204379562044|
+--------+-----------------+-----------+------------------+



**Insight**

The `groupBy()` operation was used to analyze sales by item, payment method, and location. Aggregation functions were applied to each group to identify transaction volume and sales performance.

### **12. Wide Transformations and Shuffle**

A wide transformation is an operation where data may need to move between partitions.

Examples include:

- `groupBy()`
- `dropDuplicates()`

This movement of data between partitions is called a shuffle.

Shuffle operations can be expensive because Spark needs to redistribute data across partitions.

In [31]:
sales_by_item.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[item#807], functions=[count(1), sum(total_spent#810), avg(total_spent#810)])
   +- Exchange hashpartitioning(item#807, 200), ENSURE_REQUIREMENTS, [plan_id=1406]
      +- HashAggregate(keys=[item#807], functions=[partial_count(1), partial_sum(total_spent#810), partial_avg(total_spent#810)])
         +- HashAggregate(keys=[Transaction ID#17, Price Per Unit#20, Payment Method#22, Total Spent#21, Item#18, Transaction Date#24, Quantity#19, Location#23], functions=[])
            +- Exchange hashpartitioning(Transaction ID#17, Price Per Unit#20, Payment Method#22, Total Spent#21, Item#18, Transaction Date#24, Quantity#19, Location#23, 200), ENSURE_REQUIREMENTS, [plan_id=1402]
               +- HashAggregate(keys=[Transaction ID#17, Price Per Unit#20, Payment Method#22, Total Spent#21, Item#18, Transaction Date#24, Quantity#19, Location#23], functions=[])
                  +- FileScan csv [Transaction ID#17,Item#18

**Insight**

The execution plan shows how Spark processes the grouped data. The `groupBy()` operation may require a shuffle because records with the same grouping key may be located in different partitions.

### **13. Complete Spark Data Processing Pipeline**

The complete pipeline combines the main data processing steps:

1. Load the data
2. Remove duplicates
3. Clean inconsistent values
4. Handle missing values
5. Modify the schema
6. Rename columns
7. Filter data
8. Aggregate and group data


In [32]:
# clean inconsistent data
pipeline_result = (df.dropDuplicates())

for column in text_columns:
    pipeline_result = pipeline_result.withColumn(
        column,
        when(
            trim(col(column)).isin("", "ERROR", "UNKNOWN", "NULL"),
            None
        ).otherwise(trim(col(column)))
    )

for column in numeric_columns:
    pipeline_result = pipeline_result.withColumn(
        column,
        when(
            trim(col(column)).isin("", "ERROR", "UNKNOWN", "NULL"),
            None
        ).otherwise(trim(col(column)))
    )
# handle missing values
pipeline_result = pipeline_result.fillna({
    "Item": "Unknown",
    "Payment Method": "Unknown",
    "Location": "Unknown",
    "Quantity": 0,
    "Price Per Unit": 0.0,
    "Total Spent": 0.0
})

# modify schema
pipeline_result = pipeline_result.withColumn("Quantity",
    col("Quantity").cast(IntegerType())
)

pipeline_result = pipeline_result.withColumn("Price Per Unit",
    col("Price Per Unit").cast(DoubleType())
)

pipeline_result = pipeline_result.withColumn("Total Spent",
    col("Total Spent").cast(DoubleType())
)

# raname columns
for column in pipeline_result.columns:
    pipeline_result = pipeline_result.withColumnRenamed(
        column,
        column.lower().replace(" ", "_")
    )

pipeline_result.show(5)

+--------------+-----+--------+--------------+-----------+--------------+--------+----------------+
|transaction_id| item|quantity|price_per_unit|total_spent|payment_method|location|transaction_date|
+--------------+-----+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_5307411| Cake|       3|           3.0|        0.0|Digital Wallet|Takeaway|      2023-10-21|
|   TXN_3381656| Cake|       5|           3.0|       15.0|          Cash|Takeaway|      2023-10-26|
|   TXN_6111710|Salad|       3|           5.0|       15.0|   Credit Card|In-store|      2023-06-18|
|   TXN_8715122| Cake|       2|           3.0|        6.0|          Cash|Takeaway|           ERROR|
|   TXN_1270603|Juice|       4|           3.0|       12.0|   Credit Card|Takeaway|      2023-12-03|
+--------------+-----+--------+--------------+-----------+--------------+--------+----------------+
only showing top 5 rows


**Insight**

The complete Spark pipeline successfully combines data loading, cleaning, transformation, and schema modification into a single workflow. This demonstrates how Spark DataFrames can be used to build a repeatable data processing pipeline.

### **14. Save Cleaned Data as CSV**

The cleaned DataFrame is saved in the `output` folder.

The output file will be used as the final cleaned dataset.

In [41]:
pipeline_result.toPandas().to_csv(
    "../output/clean_cafe_sales.csv",
    index=False
)

print("Clean CSV saved successfully!")

c:\Users\bajaj\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [UNSUPPORTED_PACKAGE_VERSION] PyArrow >= 18.0.0 must be installed; however, your version is 16.1.0.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Clean CSV saved successfully!


In [42]:
# verify the saved CSV
import os

print(os.path.exists("../output/clean_cafe_sales.csv"))

True


### **Final Observations and Conclusion**

The dirty café sales dataset was successfully processed using Apache Spark DataFrames.

The dataset contained missing values and inconsistent values such as `ERROR` and `UNKNOWN`. These values were cleaned and replaced with suitable values while preserving the available records.

Duplicate records were removed using `dropDuplicates()`. Numeric columns were converted to appropriate data types using casting. Column names were standardized to improve consistency.

Filtering was used to identify specific transactions. Aggregation functions such as `count()`, `sum()`, `avg()`, `min()`, and `max()` were used to calculate summary statistics.

The `groupBy()` operation was used to analyze sales by item, payment method, and location. Since `groupBy()` may require data movement between partitions, it is considered a wide transformation and may cause a shuffle.

Finally, a complete Spark data processing pipeline was created and the cleaned dataset was saved as `clean_cafe_sales.csv` in the output folder.